In [1]:
import os

# Dataset
from sklearn.datasets import fetch_covtype
from sklearn.datasets import load_wine


# Data manipulation
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:

# Results directory (relative to project root)
RESULTS_DIR = "../../results/classification"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_FILE = os.path.join(RESULTS_DIR, "classification_metrics.csv")


def save_model_results(model_name, y_true, y_pred, dataset="test"):

    results = {
        "Model": model_name,
        "Dataset": dataset,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

    df = pd.DataFrame([results])

    df.to_csv(
        RESULTS_FILE,
        mode="a",
        header=not os.path.exists(RESULTS_FILE),
        index=False
    )

    print(f"{model_name} classification metrics saved successfully!")


In [3]:
data = load_wine(as_frame=True)

X = data.data
y = data.target

print("Features:", X.shape)

display(X.head())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

Features: (178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0



Target distribution:
target
0    59
1    71
2    48
Name: count, dtype: int64


In [4]:

print("\nFeature information:")
display(X.head())


Feature information:


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [5]:
print("\nDataset description:")
print(data.DESCR[:2000])


Dataset description:
.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70

In [6]:
# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42
)

# Second split: 15% validation, 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

In [7]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (106, 13) (106,)
Validation: (36, 13) (36,)
Testing: (36, 13) (36,)


In [8]:

# Model 1: Default Logistic Regression
model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Model 2: L1 Regularisation (Lasso)
model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=1.0,
        max_iter=1000,
        random_state=42
    ))
])

# Model 3: L2 Regularisation (Ridge)
model3 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=0.1,
        max_iter=1000,
        random_state=42
    ))
])

# Model 4: Strong L1 Regularisation
model4 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.01,
        max_iter=5000,
        random_state=42
    ))
])

# Model 5: ElasticNet Regularisation
model5 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=1.0,
        l1_ratio=0.5,
        max_iter=5000,
        random_state=42
    ))
])

# Model 6: Strong L2 Regularisation
model6 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=0.01,
        max_iter=1000,
        random_state=42
    ))
])

# Model 7: Weak L2 Regularisation
model7 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=100,
        max_iter=1000,
        random_state=42
    ))
])

# Model 8: Multinomial Logistic Regression with SAGA
model8 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="saga",
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

# Model 9: Balanced Class Weights
model9 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Model 10: Different Optimisation Tolerance
model10 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=1.0,
        tol=0.0001,
        max_iter=2000,
        random_state=42
    ))
])



In [9]:

models = {
    "LR_Default": model1,
    "LR_L1": model2,
    "LR_L2_Strong": model3,
    "LR_L1_Very_Strong": model4,
    "LR_ElasticNet": model5,
    "LR_L2_Strongest": model6,
    "LR_L2_Weak": model7,
    "LR_SAGA": model8,
    "LR_Balanced": model9,
    "LR_Tolerance": model10
}

In [10]:
def train_models(models, X_train, y_train):
    for name, model in models.items():
        model.fit(X_train, y_train)
        print(f"{name} trained successfully!")


train_models(models, X_train, y_train)

LR_Default trained successfully!
LR_L1 trained successfully!
LR_L2_Strong trained successfully!
LR_L1_Very_Strong trained successfully!
LR_ElasticNet trained successfully!
LR_L2_Strongest trained successfully!
LR_L2_Weak trained successfully!
LR_SAGA trained successfully!
LR_Balanced trained successfully!
LR_Tolerance trained successfully!


/Users/rainer/Documents/CODE/Machine-Learning/.venv-ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/rainer/Documents/CODE/Machine-Learning/.venv-ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/rainer/Documents/CODE/Machine-Learning/.venv-ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leav

In [11]:
def generate_predictions(models, X):
    return {
        name: model.predict(X)
        for name, model in models.items()
    }


y_val_predictions = generate_predictions(models, X_val)
y_test_predictions = generate_predictions(models, X_test)

In [12]:

def evaluate_models(models, y_true, predictions, dataset="validation"):

    results = []

    for name in models.keys():

        y_pred = predictions[name]

        results.append({
            "Model": name,
            "Dataset": dataset,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_Weighted": precision_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "Recall_Weighted": recall_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "F1_Weighted": f1_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "F1_Macro": f1_score(
                y_true, y_pred, average="macro", zero_division=0
            )
        })

    return pd.DataFrame(results)


In [13]:
val_results = evaluate_models(
    models, y_val, y_val_predictions, "validation"
)

test_results = evaluate_models(
    models, y_test, y_test_predictions, "test"
)

print("VALIDATION RESULTS")
display(val_results)

print("TEST RESULTS")
display(test_results)

VALIDATION RESULTS


,Model,Dataset,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,F1_Macro
0,LR_Default,validation,0.972222,0.974359,0.972222,0.972174,0.972174
1,LR_L1,validation,0.972222,0.974359,0.972222,0.972174,0.972174
2,LR_L2_Strong,validation,0.972222,0.974359,0.972222,0.972174,0.972174
3,LR_L1_Very_Strong,validation,0.333333,0.111111,0.333333,0.166667,0.166667
4,LR_ElasticNet,validation,0.972222,0.974359,0.972222,0.972174,0.972174
5,LR_L2_Strongest,validation,0.972222,0.974359,0.972222,0.972174,0.972174
6,LR_L2_Weak,validation,0.972222,0.974359,0.972222,0.972174,0.972174
7,LR_SAGA,validation,0.972222,0.974359,0.972222,0.972174,0.972174
8,LR_Balanced,validation,0.972222,0.974359,0.972222,0.972174,0.972174
9,LR_Tolerance,validation,0.972222,0.974359,0.972222,0.972174,0.972174


TEST RESULTS


,Model,Dataset,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,F1_Macro
0,LR_Default,test,1.000000,1.000000,1.000000,1.000000,1.000000
1,LR_L1,test,1.000000,1.000000,1.000000,1.000000,1.000000
2,LR_L2_Strong,test,1.000000,1.000000,1.000000,1.000000,1.000000
3,LR_L1_Very_Strong,test,0.416667,0.173611,0.416667,0.245098,0.196078
4,LR_ElasticNet,test,1.000000,1.000000,1.000000,1.000000,1.000000
5,LR_L2_Strongest,test,0.972222,0.973958,0.972222,0.972156,0.976902
6,LR_L2_Weak,test,1.000000,1.000000,1.000000,1.000000,1.000000
7,LR_SAGA,test,1.000000,1.000000,1.000000,1.000000,1.000000
8,LR_Balanced,test,1.000000,1.000000,1.000000,1.000000,1.000000
9,LR_Tolerance,test,1.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
for name in models:

    save_model_results(
        name,
        y_val,
        y_val_predictions[name],
        dataset="validation"
    )

    save_model_results(
        name,
        y_test,
        y_test_predictions[name],
        dataset="test"
    )

LR_Default classification metrics saved successfully!
LR_Default classification metrics saved successfully!
LR_L1 classification metrics saved successfully!
LR_L1 classification metrics saved successfully!
LR_L2_Strong classification metrics saved successfully!
LR_L2_Strong classification metrics saved successfully!
LR_L1_Very_Strong classification metrics saved successfully!
LR_L1_Very_Strong classification metrics saved successfully!
LR_ElasticNet classification metrics saved successfully!
LR_ElasticNet classification metrics saved successfully!
LR_L2_Strongest classification metrics saved successfully!
LR_L2_Strongest classification metrics saved successfully!
LR_L2_Weak classification metrics saved successfully!
LR_L2_Weak classification metrics saved successfully!
LR_SAGA classification metrics saved successfully!
LR_SAGA classification metrics saved successfully!
LR_Balanced classification metrics saved successfully!
LR_Balanced classification metrics saved successfully!
LR_Toler

In [15]:
def extract_coefficients(models, feature_names=None):

    results = []

    for name, model in models.items():

        for i, coefficient in enumerate(model.coef_):

            results.append({
                "Model": name,
                "Feature": feature_names[i] if feature_names is not None else f"Feature_{i}",
                "Coefficient": coefficient
            })

    return pd.DataFrame(results)

In [16]:

def extract_coefficients(models, feature_names):

    results = []

    for name, model in models.items():

        # Extract the final estimator from the Pipeline
        classifier = model.named_steps["model"]

        for i, coefficients in enumerate(classifier.coef_):

            for feature, coefficient in zip(feature_names, coefficients):

                results.append({
                    "Model": name,
                    "Class": classifier.classes_[i],
                    "Feature": feature,
                    "Coefficient": coefficient
                })

    return pd.DataFrame(results)


coefficient_results = extract_coefficients(
    models,
    X_train.columns
)

display(coefficient_results)


,Model,Class,Feature,Coefficient
0,LR_Default,0,alcohol,0.691387
1,LR_Default,0,malic_acid,0.104593
2,LR_Default,0,ash,0.358783
3,LR_Default,0,alcalinity_of_ash,-0.862905
4,LR_Default,0,magnesium,0.168269
...,...,...,...,...
385,LR_Tolerance,2,proanthocyanins,-0.576044
386,LR_Tolerance,2,color_intensity,0.630479
387,LR_Tolerance,2,hue,-0.783850
388,LR_Tolerance,2,od280/od315_of_diluted_wines,-0.749534


In [18]:
all_results = pd.concat(
    [val_results, test_results],
    ignore_index=True
)

display(all_results)

,Model,Dataset,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,F1_Macro
0,LR_Default,validation,0.972222,0.974359,0.972222,0.972174,0.972174
1,LR_L1,validation,0.972222,0.974359,0.972222,0.972174,0.972174
2,LR_L2_Strong,validation,0.972222,0.974359,0.972222,0.972174,0.972174
3,LR_L1_Very_Strong,validation,0.333333,0.111111,0.333333,0.166667,0.166667
4,LR_ElasticNet,validation,0.972222,0.974359,0.972222,0.972174,0.972174
5,LR_L2_Strongest,validation,0.972222,0.974359,0.972222,0.972174,0.972174
6,LR_L2_Weak,validation,0.972222,0.974359,0.972222,0.972174,0.972174
7,LR_SAGA,validation,0.972222,0.974359,0.972222,0.972174,0.972174
8,LR_Balanced,validation,0.972222,0.974359,0.972222,0.972174,0.972174
9,LR_Tolerance,validation,0.972222,0.974359,0.972222,0.972174,0.972174


# Logistic Regression: Data Science Interview Preparation

---

# 1. Logistic Regression Fundamentals

### Q1. What is Logistic Regression?

Logistic Regression is a supervised machine learning algorithm used to predict a categorical target variable. It models the probability that an observation belongs to a class using a weighted sum of the input features passed through a sigmoid (or softmax) function.

### Q2. What is the equation of Logistic Regression?

For binary classification, the model first calculates a linear score:

$$
z = \beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_nx_n
$$

It then converts the score into a probability:

$$
P(y=1|X)=\frac{1}{1+e^{-z}}
$$

Where:

- $z$: Linear score (log-odds).
- $\beta_0$: Intercept (bias).
- $\beta_i$: Coefficients (weights).
- $x_i$: Independent variables (features).

### Q3. Why is it called "regression" if it is used for classification?

The model is a regression on the log-odds of the outcome: it fits a linear function of the features to $\log\frac{p}{1-p}$. The final class label is obtained afterwards by applying a probability threshold to the predicted probability.

### Q4. What is the difference between Binary, Multinomial and Ordinal Logistic Regression?

- **Binary Logistic Regression:** Two possible classes (for example, churn / no churn).
- **Multinomial Logistic Regression:** Three or more unordered classes (for example, the three Wine cultivars).
- **Ordinal Logistic Regression:** Three or more classes with a natural order (for example, low / medium / high). It is not provided by Scikit-learn's `LogisticRegression`.

### Q5. How does Logistic Regression learn its coefficients?

It uses Maximum Likelihood Estimation (MLE), which is equivalent to minimising the log loss (binary cross-entropy). Unlike OLS, there is no closed-form solution, so an iterative numerical optimiser is required.

### Q6. What is the difference between Linear Regression and Logistic Regression?

| Aspect | Linear Regression | Logistic Regression |
|---|---|---|
| Target | Continuous | Categorical |
| Output | Any real number | Probability between 0 and 1 |
| Link to features | Linear in the expected value | Linear in the log-odds |
| Loss function | Mean Squared Error | Log loss |
| Estimation | OLS (closed form) | MLE (iterative solver) |
| Typical metrics | MAE, RMSE, R² | Precision, recall, F1, ROC-AUC |

### Q7. What is a decision boundary?

The decision boundary is the set of points where the predicted probability equals the classification threshold. With the default threshold of 0.5, the boundary is where $z=0$, which is a linear (hyperplane) boundary in the feature space.

---

# 2. Mathematical and Algorithmic Understanding

### Q8. What are odds and log-odds?

$$
\text{Odds}=\frac{p}{1-p}
\qquad
\text{Log-odds (logit)}=\log\left(\frac{p}{1-p}\right)
$$

Logistic Regression assumes the log-odds are a linear function of the features:

$$
\log\left(\frac{p}{1-p}\right)=\beta_0+\beta_1x_1+\cdots+\beta_nx_n
$$

### Q9. What is the sigmoid function and what are its properties?

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

- Output always lies between 0 and 1.
- $\sigma(0)=0.5$.
- Large positive $z$ gives probabilities close to 1.
- Large negative $z$ gives probabilities close to 0.
- It is smooth and differentiable, with $\sigma'(z)=\sigma(z)(1-\sigma(z))$.

### Q10. What is the cost function of Logistic Regression?

The log loss (binary cross-entropy):

$$
J(\beta)=-\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{p}_i)+(1-y_i)\log(1-\hat{p}_i)\right]
$$

Confident wrong predictions are penalised very heavily.

### Q11. Why don't we use Mean Squared Error for Logistic Regression?

- Combined with the sigmoid function, MSE produces a non-convex loss surface with flat regions, making optimisation harder.
- Log loss is convex for Logistic Regression, so the optimiser finds the global minimum.
- Log loss is the negative log-likelihood of the Bernoulli model, so it has a direct probabilistic justification.

### Q12. How does Gradient Descent work for Logistic Regression?

Parameters are updated iteratively in the direction that reduces the log loss:

$$
\beta_{new}=\beta_{old}-\alpha\frac{\partial J}{\partial\beta}
$$

For Logistic Regression, the gradient has a simple form:

$$
\frac{\partial J}{\partial\beta_j}=\frac{1}{n}\sum_{i=1}^{n}(\hat{p}_i-y_i)x_{ij}
$$

Where $\alpha$ is the learning rate.

### Q13. Does Logistic Regression have a closed-form solution?

No. Unlike OLS Linear Regression, the MLE equations are nonlinear in the coefficients. Scikit-learn uses numerical solvers such as `lbfgs`, `newton-cg`, `liblinear`, `sag` and `saga`.

### Q14. What is the softmax function?

For $K$ classes, softmax converts $K$ scores into a probability distribution:

$$
P(y=k|X)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

Probabilities are positive and sum to 1. It is the multiclass generalisation of the sigmoid.

### Q15. What is the difference between Multinomial and One-vs-Rest (OvR)?

- **Multinomial:** Optimises one joint softmax loss across all classes.
- **One-vs-Rest:** Trains one separate binary classifier per class (class vs everything else) and picks the class with the highest score.

In modern Scikit-learn, multinomial classification is used automatically for three or more classes with supported solvers.

### Q16. How do you interpret the coefficients?

A one-unit increase in $x_j$, holding other features constant, changes the log-odds by $\beta_j$. Exponentiating gives the odds ratio:

$$
\text{Odds ratio}=e^{\beta_j}
$$

- $e^{\beta_j}>1$: Higher odds of the positive class.
- $e^{\beta_j}<1$: Lower odds of the positive class.
- $e^{\beta_j}=1$: No effect.

Coefficients are only directly comparable across features if the features are on the same scale.

---

# 3. Assumptions of Logistic Regression

### Q17. What are the main assumptions of Logistic Regression?

| Assumption | Meaning |
|---|---|
| Appropriate outcome | Target is binary (or categorical for multinomial) |
| Independence | Observations are independent of each other |
| Linearity in the logit | Log-odds are a linear function of the features |
| No perfect multicollinearity | No predictor is an exact linear combination of others |
| Sufficient sample size | Enough observations per class and per predictor |
| No extreme influential outliers | Individual points should not dominate the fit |

### Q18. Which Linear Regression assumptions does Logistic Regression NOT require?

- Normally distributed residuals.
- Homoscedasticity (constant error variance).
- A linear relationship between the features and the target itself (only the log-odds need to be linear).

### Q19. What happens if these assumptions are violated?

- **Non-linear logit:** Model may underfit; consider transformations, splines or interaction terms.
- **Dependent observations:** Standard errors and inference become unreliable (for example repeated measures per customer).
- **Multicollinearity:** Coefficient estimates become unstable and hard to interpret.
- **Small sample size:** Coefficients can be unstable or overfit.
- **Influential outliers:** Can distort the fitted boundary.

Not every violation makes predictions unusable. Some assumptions primarily affect statistical inference.

### Q20. How do you detect multicollinearity?

- Correlation matrix.
- Variance Inflation Factor (VIF).
- Condition number.

The same VIF heuristics apply as in Linear Regression: above 5 is worth investigating, above 10 is often considered concerning.

### Q21. How do you check the linearity-of-logit assumption?

- Box-Tidwell test.
- Plot binned feature values against the empirical log-odds.
- Compare the model against one with polynomial or spline terms.

### Q22. What is perfect (complete) separation?

Perfect separation occurs when a feature, or a combination of features, perfectly splits the classes. Unregularised MLE then pushes coefficients towards infinity and the solver fails to converge.

Fixes:

- Apply regularisation (smaller `C`).
- Remove or combine the offending feature.
- Collect more data.

---

# 4. Classification Evaluation Metrics

### Q23. What is a confusion matrix?

A table comparing actual and predicted classes.

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

### Q24. What is Accuracy?

$$
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
$$

It can be misleading on imbalanced datasets.

### Q25. What are Precision, Recall and F1-score?

$$
\text{Precision}=\frac{TP}{TP+FP}
\qquad
\text{Recall}=\frac{TP}{TP+FN}
$$

$$
F_1=2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}
$$

Example with TP = 3, FP = 3, TN = 2, FN = 2:

- Precision = 3 / (3 + 3) = 50%.
- Recall = 3 / (3 + 2) = 60%.

Interpretation:

- **Precision:** Of everything predicted positive, how much was correct?
- **Recall:** Of all actual positives, how many were found?

### Q26. What is Specificity?

$$
\text{Specificity}=\frac{TN}{TN+FP}
$$

The proportion of actual negatives correctly identified.

### Q27. What is ROC-AUC?

The ROC curve plots True Positive Rate (recall) against False Positive Rate across all thresholds. AUC is the area under that curve.

- AUC = 1: Perfect ranking.
- AUC = 0.5: No better than random ranking.
- AUC below 0.5: Worse than random (predictions are inverted).

AUC is threshold-independent and measures how well the model ranks positives above negatives.

### Q28. When would you prefer Precision-Recall AUC over ROC-AUC?

When the positive class is rare. ROC-AUC can look optimistic on heavily imbalanced data because the large number of true negatives keeps the false positive rate small. PR-AUC focuses on the positive class.

### Q29. What is log loss as an evaluation metric?

Log loss evaluates the quality of predicted probabilities, not just the final labels. Confident wrong predictions are penalised heavily. Lower is better.

### Q30. How does the classification threshold work?

The default threshold is 0.5: predict class 1 if $\hat{p}\ge0.5$.

- **Lower threshold:** More positives predicted, higher recall, lower precision.
- **Higher threshold:** Fewer positives predicted, higher precision, lower recall.

The threshold is not a training parameter of `LogisticRegression`. It is chosen after training, based on the cost of false positives versus false negatives.

### Q31. What is the accuracy paradox?

A model can achieve high accuracy by always predicting the majority class. For example, if 99% of transactions are legitimate, predicting "legitimate" for everything gives 99% accuracy but zero fraud detected.

### Q32. How do you evaluate multiclass classification?

Use per-class precision, recall and F1, aggregated by:

- **Macro average:** Unweighted mean across classes.
- **Weighted average:** Mean weighted by class support.
- **Micro average:** Pools all TP, FP and FN across classes.

Also inspect the full confusion matrix to see which classes are being confused.

### Q33. What is probability calibration?

A model is well calibrated if predicted probabilities match observed frequencies (among predictions of 0.8, roughly 80% should be positive). Logistic Regression is often reasonably calibrated, but `class_weight="balanced"` and strong regularisation can distort probabilities. Use calibration curves and, if required, `CalibratedClassifierCV`.

---

# 5. Overfitting, Underfitting and Regularisation

### Q34. What are overfitting and underfitting in Logistic Regression?

- **Overfitting:** The model learns training-specific noise, often with very large coefficients, and performs poorly on unseen data.
- **Underfitting:** The model is too simple or too heavily regularised to capture the pattern.

### Q35. What is regularisation?

Regularisation adds a penalty on coefficient size to the loss function to control model complexity and reduce overfitting.

### Q36. What is the difference between L1 and L2 regularisation?

**L2 (Ridge):**

$$
Loss=LogLoss+\lambda\sum_{j=1}^{n}w_j^2
$$

- Shrinks coefficients towards zero.
- Generally does not make coefficients exactly zero.
- Default in Scikit-learn.

**L1 (Lasso):**

$$
Loss=LogLoss+\lambda\sum_{j=1}^{n}|w_j|
$$

- Can force coefficients to exactly zero.
- Produces sparse models and performs implicit feature selection.
- Requires a solver that supports L1 (`liblinear` or `saga`).

### Q37. What is ElasticNet regularisation?

ElasticNet combines L1 and L2:

$$
Loss=LogLoss+\lambda\left[r\sum|w_j|+(1-r)\sum w_j^2\right]
$$

It shrinks coefficients and can eliminate some features. In Scikit-learn it requires `solver="saga"` and the `l1_ratio` parameter.

### Q38. Why does a smaller C mean stronger regularisation?

`C` is the inverse of the regularisation strength:

$$
C=\frac{1}{\lambda}
$$

| C | Regularisation | Expected behaviour |
|---|---|---|
| 0.001 | Extremely strong | Coefficients heavily restricted |
| 0.01 | Very strong | Strong coefficient shrinkage |
| 0.1 | Strong | More restricted model |
| 1 | Default | Standard regularisation |
| 10 | Weak | More flexibility |
| 100 | Very weak | Less coefficient restriction |
| 1000 | Extremely weak | Higher risk of overfitting |

### Q39. What happens if C is extremely large?

The penalty becomes negligible and the model approaches an unregularised fit. Coefficients can grow large, variance increases and overfitting becomes more likely. A higher `C` does not automatically mean better performance.

### Q40. Why do we need feature scaling for Logistic Regression?

- Regularisation penalises coefficient magnitudes, so features on larger scales receive unequal effective penalties.
- Gradient-based solvers (`lbfgs`, `sag`, `saga`) converge faster and more reliably on standardised features.
- Standardised coefficients are easier to compare.

Fit the scaler only on training data (preferably inside a `Pipeline`).

### Q41. Why does L1 regularisation produce zero coefficients?

The L1 penalty has a "corner" at zero. When the gradient of the loss for a feature is smaller than the penalty strength, the optimal coefficient sits exactly at zero. The L2 penalty has no such corner, so it shrinks coefficients smoothly without eliminating them.

---

# 6. Practical Python and Scikit-learn

### Q42. How do you train a Logistic Regression model?

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
```

### Q43. How do you extract coefficients and intercept?

```python
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
```

For a multiclass problem, `coef_` has shape `(n_classes, n_features)`.

### Q44. How do you evaluate a classification model?

```python
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Log loss:", log_loss(y_test, y_proba))

# Binary: roc_auc_score(y_test, y_proba[:, 1])
# Multiclass:
print("ROC-AUC:", roc_auc_score(y_test, y_proba, multi_class="ovr"))
```

### Q45. What are the important hyperparameters of Scikit-learn LogisticRegression?

```python
LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=100,
    tol=1e-4,
    class_weight=None,
    fit_intercept=True,
    random_state=None,
    dual=False,
    intercept_scaling=1,
    l1_ratio=None,
    verbose=0,
    warm_start=False
)
```

They fall into three groups:

1. **Regularisation:** `penalty`, `C`, `l1_ratio`.
2. **Optimisation:** `solver`, `max_iter`, `tol`, `warm_start`, `random_state`.
3. **Configuration:** `class_weight`, `fit_intercept`, `dual`, `intercept_scaling`, `verbose`.

`C`, `penalty` and `solver` change how the model is trained. `max_iter` and `tol` primarily control how the optimisation process runs.

### Q46. What values can `penalty` take?

| Value | Meaning |
|---|---|
| `'l2'` | Ridge regularisation (default) |
| `'l1'` | Lasso regularisation |
| `'elasticnet'` | Combination of L1 and L2 |
| `None` | No regularisation |

```python
LogisticRegression(penalty="l1", solver="saga")
LogisticRegression(penalty="l2", solver="lbfgs")
LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5)
LogisticRegression(penalty=None, solver="lbfgs")
```

Version note: in recent Scikit-learn releases (1.8 onwards) `penalty` is being deprecated in favour of specifying `l1_ratio` and `C`. Check your installed version before mixing the two conventions.

### Q47. What are the available solvers and when should you use each?

| Solver | How it works | Best use |
|---|---|---|
| `lbfgs` | Quasi-Newton optimisation | General-purpose default |
| `liblinear` | Coordinate descent / trust-region | Small binary datasets |
| `newton-cg` | Newton optimisation | Multiclass problems with L2 |
| `newton-cholesky` | Newton method using the Hessian | Many samples, relatively few features, binary |
| `sag` | Stochastic Average Gradient | Large datasets |
| `saga` | Improved SAG with proximal updates | L1, L2, ElasticNet on large datasets |

### Q48. Which solvers support which penalties?

| Solver | L1 | L2 | ElasticNet | None | Multinomial |
|---|---|---|---|---|---|
| `lbfgs` | No | Yes | No | Yes | Yes |
| `liblinear` | Yes | Yes | No | No | No* |
| `newton-cg` | No | Yes | No | Yes | Yes |
| `newton-cholesky` | No | Yes | No | Yes | No* |
| `sag` | No | Yes | No | Yes | Yes |
| `saga` | Yes | Yes | Yes | Yes | Yes |

\*Can be extended to multiclass through One-vs-Rest, for example `OneVsRestClassifier`.

For the Wine dataset, `lbfgs` and `saga` are particularly useful choices.

### Q49. What is the difference between `lbfgs` and `saga`?

- **`lbfgs`:** Quasi-Newton method. Fast and reliable on small to medium datasets. Supports L2 and no penalty only.
- **`saga`:** Stochastic gradient method. Supports L1, L2 and ElasticNet, scales well to large datasets, but converges more slowly and strongly benefits from feature scaling.

### Q50. Why does `liblinear` fail on a three-class dataset like Wine?

`liblinear` does not support the multinomial loss. Depending on the version, it either raises an error or falls back to a One-vs-Rest scheme. Use `lbfgs`, `newton-cg`, `sag` or `saga` for multinomial classification, or wrap `liblinear` in `OneVsRestClassifier`.

### Q51. What is the difference between `max_iter` and `tol`?

- **`max_iter`:** The maximum number of iterations the solver can perform (default 100).
- **`tol`:** How strict the stopping condition is (default 0.0001). Smaller values require a tighter convergence.

They work together: the solver stops when either the tolerance is satisfied or `max_iter` is reached.

| `max_iter` | Meaning |
|---|---|
| 100 | Default |
| 500 | More opportunity to converge |
| 1000 | Higher iteration limit |
| 5000 | Very high iteration limit |

| `tol` | Meaning |
|---|---|
| 0.1 | Loose tolerance |
| 0.01 | Less strict |
| 0.001 | More strict |
| 0.0001 | Default |
| 0.00001 | Very strict |

### Q52. What does `ConvergenceWarning: lbfgs failed to converge` mean?

The solver reached `max_iter` without satisfying its convergence criterion. Possible fixes:

- Increase `max_iter` (for example 1000).
- Scale the features with `StandardScaler`.
- Try a different solver.
- Increase regularisation (smaller `C`), especially if perfect separation is present.

Increasing `max_iter` allows the solver to converge, but it does not necessarily improve model performance.

### Q53. How does `class_weight` work?

It controls how much importance each class receives in the loss function.

```python
class_weight=None           # all observations weighted equally
class_weight="balanced"     # weights inversely proportional to class frequency
class_weight={0: 1, 1: 2, 2: 3}   # custom weights
```

The balanced formula is:

$$
w_i=\frac{N}{K\times N_i}
$$

Where $N$ is the total number of observations, $K$ is the number of classes and $N_i$ is the number of observations in class $i$.

For the Wine dataset, try both `None` and `"balanced"`, but do not assume balanced is better: the three classes are not extremely imbalanced.

### Q54. What does `fit_intercept` do?

It decides whether the model learns the intercept $b$. Keep it `True` unless you have a specific mathematical reason, such as properly centred data where a zero intercept is deliberately required.

### Q55. What does `random_state` control?

It makes random operations reproducible. In `LogisticRegression` it only affects `sag`, `saga` and `liblinear`. It has no effect on `lbfgs`. The number 42 has no special mathematical significance.

### Q56. What are `dual` and `intercept_scaling`?

- **`dual`:** Chooses between the primal and dual optimisation formulations. It is only supported for L2 with `liblinear`. Use `dual=False` when there are more samples than features.
- **`intercept_scaling`:** Only relevant for `liblinear` with `fit_intercept=True`. It scales the synthetic feature used to represent the intercept, reducing regularisation on the intercept as it grows. Leave at 1 in most cases.

### Q57. What is `l1_ratio`?

The ElasticNet mixing parameter, ranging from 0 to 1.

| Value | Regularisation |
|---|---|
| 0 | Pure L2 |
| 0.25 | 25% L1 + 75% L2 |
| 0.5 | 50% L1 + 50% L2 |
| 0.75 | 75% L1 + 25% L2 |
| 1 | Pure L1 |

```python
LogisticRegression(solver="saga", penalty="elasticnet", l1_ratio=0.5, C=1.0)
```

### Q58. What are `warm_start`, `verbose` and `n_jobs`?

- **`warm_start`:** If `True`, the next `.fit()` reuses the previous coefficients as the starting point. Supported by every solver except `liblinear`. It does not automatically make every fit faster.
- **`verbose`:** Controls how much the solver prints during training. Useful for monitoring long runs.
- **`n_jobs`:** Deprecated and has no effect in current versions, so it can be ignored.

### Q59. What happened to `multi_class`?

`multi_class` (`"ovr"` or `"multinomial"`) is deprecated in modern Scikit-learn. Multinomial classification is applied automatically for three or more classes with supported solvers, so it is best omitted. For One-vs-Rest behaviour, use `OneVsRestClassifier`.

### Q60. How do you combine scaling and Logistic Regression correctly?

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

pipe.fit(X_train, y_train)
```

The scaler is fitted only on training data, which prevents data leakage during cross-validation.

### Q61. How do you tune hyperparameters?

```python
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro")
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
```

For L1 or ElasticNet, vary `solver="saga"` and the penalty parameters in a separate grid because not every combination is valid.

### Q62. Which experiments are worth running on the Wine dataset?

Organise experiments in groups instead of changing everything at once. Wrap each model in a `StandardScaler` pipeline.

| Experiment | Configuration | What you learn |
|---|---|---|
| 1. Baseline | `C=1.0`, `lbfgs` | Default behaviour |
| 2. Strong regularisation | `C=0.01`, `lbfgs` | Effect of shrinkage |
| 3. Weak regularisation | `C=100`, `lbfgs` | Risk of overfitting |
| 4. L1 | `penalty="l1"`, `solver="saga"` | Sparse coefficients |
| 5. ElasticNet | `solver="saga"`, `l1_ratio=0.5` | L1 and L2 trade-off |
| 6. No regularisation | `penalty=None`, `lbfgs` | Unpenalised baseline |
| 7. Balanced weights | `class_weight="balanced"` | Class weighting effect |
| 8. Iteration limit | `max_iter=50` vs `1000` | Convergence warnings |
| 9. Tolerance | `tol=0.1` vs `0.00001` | Stopping strictness |
| 10. Solver comparison | `lbfgs` vs `saga` vs `newton-cg` | Same result, different path |

Always check your Scikit-learn version first, because the penalty API changed in recent releases:

```python
import sklearn
print(sklearn.__version__)
```

---

# 7. Practical Data Science Questions

### Q63. You have a dataset containing missing values. How would you handle them before training Logistic Regression?

1. Identify missing values using `isnull()`.
2. Understand why values are missing.
3. Check the percentage of missing values per feature.
4. Decide whether to drop or impute.
5. Use median imputation for numerical variables where appropriate.
6. Use mode or a separate category for categorical variables.

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)
```

Important: Fit the imputer only on training data to avoid data leakage. Logistic Regression cannot handle missing values natively.

### Q64. Your dataset contains categorical variables. How would you use them?

Logistic Regression requires numerical input. Use:

- One-Hot Encoding for unordered categories.
- Ordinal Encoding where categories have a genuine order.

```python
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", drop="first")
```

`drop="first"` removes a reference category, which avoids perfect multicollinearity when an intercept is included. With regularisation, dropping is optional.

### Q65. Your dataset contains outliers. What would you do?

1. Identify outliers using boxplots, IQR or domain knowledge.
2. Investigate whether they are genuine observations or errors.
3. Check their influence on the fitted model.
4. Consider winsorising, transformations, or robust scaling.

Do not automatically remove every outlier.

### Q66. Your numerical features have very different scales. What would you do?

Standardise them inside a pipeline:

$$
z=\frac{x-\mu}{\sigma}
$$

Scaling matters more for Logistic Regression than for OLS because of the regularisation penalty and the iterative solvers.

### Q67. Your classes are heavily imbalanced. What would you do?

- Use appropriate metrics: precision, recall, F1, PR-AUC.
- Use `class_weight="balanced"`.
- Tune the decision threshold.
- Use stratified splits and stratified cross-validation.
- Consider resampling (SMOTE, undersampling), applied only within training folds.
- Collect more minority-class data.

### Q68. Your model has 99% accuracy on a dataset where 99% of the observations are in one class. Is it a good model?

Not necessarily. A model that always predicts the majority class achieves the same accuracy. Check the confusion matrix, recall for the minority class, PR-AUC, and compare against a dummy baseline.

### Q69. Your model has a training accuracy of 0.99 but test accuracy of 0.70. What would you investigate?

- Overfitting (try a smaller `C`).
- Data leakage.
- Distribution shift between train and test.
- Too many irrelevant features.
- Insufficient training data.
- Different preprocessing between training and test sets.

Possible solutions: stronger regularisation, feature selection, better validation, more representative data.

### Q70. You get a `ConvergenceWarning` after training. What do you do?

Scale the features, increase `max_iter`, try another solver, and check for perfect separation or extremely large `C`. Do not simply ignore the warning, because the coefficients may not be the optimal solution.

### Q71. Two independent variables have a correlation of 0.99. What would you do?

Investigate multicollinearity. Possible approaches:

- Remove redundant features.
- Combine related features.
- Use PCA.
- Use L2 regularisation (a smaller `C`).

The choice depends on whether prediction or coefficient interpretability is the main goal.

### Q72. You have 100 features but only 50 observations. What would you do?

This is a high-dimensional, small-sample problem. Unregularised Logistic Regression will likely overfit or hit perfect separation.

- Use L1 or ElasticNet regularisation.
- Use a small `C`.
- Apply feature selection.
- Use cross-validation.
- Collect more observations.

### Q73. How would you choose a classification threshold?

1. Get predicted probabilities with `predict_proba`.
2. Define the cost of false positives versus false negatives.
3. Plot the precision-recall curve (or ROC curve).
4. Pick the threshold that optimises the business metric (for example maximum F1, or recall of at least 90%).
5. Select the threshold on validation data, not on test data.

### Q74. How would you identify the most important features?

- Standardised coefficients.
- Odds ratios.
- Features retained by L1 regularisation.
- Permutation importance.
- Confidence intervals (for example with `statsmodels`).

Important: A large coefficient does not automatically mean the feature is more important. Feature scales, multicollinearity and uncertainty all matter.

### Q75. Your model performs well on training and validation data but poorly on testing data. What could be the reason?

- Distribution shift.
- Test data from a different time period.
- Previously unseen categories.
- Data leakage during model selection.
- Small or unrepresentative test set.

---

# 8. Real-World Data Science Interview Scenarios

These are more open-ended questions. Interviewers expect you to explain your approach rather than give one fixed answer.

### Q76. A telecom company wants to predict customer churn. How would you build a model?

Expected approach:

1. Define churn and the prediction horizon.
2. Understand the dataset and target imbalance.
3. Perform exploratory data analysis.
4. Handle missing values and outliers.
5. Engineer features (tenure, usage trends, support calls).
6. Encode categorical variables.
7. Use stratified train, validation and test splits.
8. Build a Logistic Regression baseline.
9. Evaluate using precision, recall, F1 and PR-AUC.
10. Tune `C` and `class_weight`; tune the threshold.
11. Compare against tree-based models.
12. Interpret odds ratios and report test performance.

### Q77. A bank wants to detect fraudulent transactions. How would you approach this?

Important considerations:

- Extreme class imbalance, so avoid accuracy.
- Use PR-AUC, recall at a fixed precision, and cost-based evaluation.
- Time-based splitting to avoid leakage.
- `class_weight` or resampling.
- Threshold chosen from the cost of missed fraud versus false alarms.
- Monitor for drift, because fraud patterns change.

### Q78. A hospital wants a model to predict whether a patient has a disease. What would you consider?

- Recall (sensitivity) is usually prioritised, since missing a sick patient is costly.
- Calibrated probabilities are important for risk communication.
- Interpretability (odds ratios) matters for clinical trust.
- Use a lower decision threshold if needed.
- Check performance across patient subgroups.
- Validate externally on data from another hospital if possible.

### Q79. You are building a spam classifier. How would you use Logistic Regression?

- Convert text to numerical features using TF-IDF or bag-of-words.
- The feature matrix is sparse and high-dimensional, so use `saga` or `liblinear` with L1 or L2.
- Tune `C` using cross-validation.
- Evaluate precision (avoid flagging real emails) and recall.
- Inspect the largest coefficients to understand which words drive predictions.

### Q80. How would you approach the Wine classification project?

1. Load the dataset (178 samples, 13 features, 3 cultivars) and run EDA.
2. Split the data using a stratified train and test split.
3. Build a `StandardScaler` and `LogisticRegression` pipeline.
4. Run the experiment plan from Q62 (`C`, penalty, solver, `max_iter`, `tol`, `class_weight`).
5. Use stratified cross-validation to compare configurations.
6. Evaluate with accuracy, macro F1 and the confusion matrix.
7. Inspect coefficients per class and see which features matter.
8. Report final test performance only once.

### Q81. A stakeholder asks why the model predicted "yes" for a specific customer. How do you explain it?

Use the coefficients: each feature contributes $\beta_jx_j$ to the log-odds. Describe the largest contributions, translate coefficients to odds ratios, and show how the probability would change if a feature changed. This transparency is a major advantage of Logistic Regression.

### Q82. When would you choose Logistic Regression over a more complex model?

- Interpretability and explainability are required.
- The dataset is small or medium sized.
- A fast, strong baseline is needed.
- Probabilities need to be reasonably calibrated.
- The relationship is approximately linear in the log-odds.
- Low latency and low maintenance matter.

Prefer tree-based or other nonlinear models when there are strong feature interactions or nonlinear patterns that cannot be captured with feature engineering.

---

# 9. Interview Questions You Should Be Able to Answer

1. What is the difference between L1 and L2 regularisation?
2. Why does a smaller `C` result in stronger regularisation?
3. What happens if `C` is extremely large?
4. Why is feature scaling important for Logistic Regression?
5. What is the difference between `lbfgs` and `saga`?
6. Why does `liblinear` fail with a three-class dataset like Wine?
7. What happens when `max_iter` is too small?
8. What is the difference between `max_iter` and `tol`?
9. How does `class_weight="balanced"` work?
10. Why might L1 regularisation produce zero coefficients?
11. What is ElasticNet, and why combine L1 and L2?
12. What is the difference between multinomial and One-vs-Rest classification?
13. Why is log loss used instead of MSE?
14. How do you interpret a coefficient as an odds ratio?
15. How do you choose a classification threshold?
16. What is perfect separation and how do you fix it?

---

# 10. Quick Revision Sheet

| Concept | Remember |
|---|---|
| Logistic Regression | Models class probability using sigmoid or softmax of a linear score |
| Sigmoid | Maps any real number to a value between 0 and 1 |
| Logit | Log-odds, the quantity that is linear in the features |
| Odds ratio | $e^{\beta_j}$, multiplicative change in odds per unit increase |
| MLE | Estimation method, equivalent to minimising log loss |
| Log loss | Penalises confident wrong probabilities |
| Softmax | Multiclass generalisation of sigmoid |
| Multinomial | Joint loss across all classes |
| OvR | One binary classifier per class |
| Decision boundary | Where $z=0$ at a threshold of 0.5, which is linear |
| Threshold | Chosen after training, trades precision against recall |
| Precision | $TP/(TP+FP)$ |
| Recall | $TP/(TP+FN)$ |
| F1 | Harmonic mean of precision and recall |
| ROC-AUC | Threshold-independent ranking quality |
| PR-AUC | Preferred for imbalanced data |
| L2 | Shrinks coefficients, default |
| L1 | Sparse coefficients, feature selection |
| ElasticNet | Combines L1 and L2, needs `saga` and `l1_ratio` |
| `C` | Inverse regularisation strength, smaller means stronger |
| `solver` | Optimisation algorithm, not all support every penalty |
| `max_iter` | Maximum solver iterations |
| `tol` | Convergence strictness |
| `class_weight` | Reweights classes in the loss |
| `liblinear` | Small binary problems, no multinomial |
| `lbfgs` | Default, L2 or none, multinomial |
| `saga` | L1, L2, ElasticNet, multinomial, scaling needed |
| Perfect separation | Coefficients diverge, fix with regularisation |
| Training set | Learn |
| Validation set | Decide |
| Test set | Verify |